# 11 — Assess learners

Leaf-size overfitting, bagging, decision trees vs random trees, InsaneLearner, and synthetic best-for-* datasets.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantlab.experiments import leaf_size_rmse_curve, train_test_split_shuffle, best_for_linear_regression, best_for_decision_tree, rmse
from quantlab.experiments.assess_learners import compare_dt_rt_timing
from quantlab.learners import LinearRegressionLearner, RegressionTree, InsaneLearner, BaggedTrees

df = pd.read_csv(Path('..') / 'data' / 'learners' / 'Istanbul.csv')
data = df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
X, y = data[:, :-1], data[:, -1]
Xtr, Xte, ytr, yte = train_test_split_shuffle(X, y, seed=0)
print(X.shape)


In [ ]:
curve = leaf_size_rmse_curve(Xtr, ytr, Xte, yte, leaf_sizes=range(1, 81, 2), seed=0)
bag = leaf_size_rmse_curve(Xtr, ytr, Xte, yte, leaf_sizes=range(1, 81, 2), bag=True, n_estimators=10, seed=0)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(curve['leaf_size'], curve['in_rmse'], label='in'); axes[0].plot(curve['leaf_size'], curve['out_rmse'], label='out'); axes[0].legend(); axes[0].set_title('DT')
axes[1].plot(bag['leaf_size'], bag['in_rmse'], label='in'); axes[1].plot(bag['leaf_size'], bag['out_rmse'], label='out'); axes[1].legend(); axes[1].set_title('Bagged DT')
plt.show()


In [ ]:
Xl, yl = best_for_linear_regression(); Xd, yd = best_for_decision_tree()
for name, Xb, yb in [('favors LinReg', Xl, yl), ('favors DT', Xd, yd)]:
    Xtr, Xte, ytr, yte = train_test_split_shuffle(Xb, yb, seed=1)
    lr = LinearRegressionLearner().fit(Xtr, ytr)
    dt = RegressionTree(leaf_size=1).fit(Xtr, ytr)
    print(name, 'LR', rmse(yte, lr.predict(Xte)), 'DT', rmse(yte, dt.predict(Xte)))
ins = InsaneLearner(n_outer=5, n_inner=5, seed=0).fit(Xtr, ytr)
print('InsaneLearner RMSE', rmse(yte, ins.predict(Xte)))
